In [118]:
from csv import DictReader
from functools import partial
from itertools import islice
import wrapt
from dateutil import parser

In [7]:
with open("./data/sample_data.csv") as f:
    f = f.read()
    lines = f.splitlines()
    for line in lines:
        print(line.split(','))

['id', 'name', 'signup_date', 'score', 'active']
['1', 'Alice', '2023-01-15', '85.5', 'True']
['2', 'Bob', '2023-03-22', '92.0', 'False']
['3', 'Charlie', '2023-05-10', '78.0', 'True']
['4', 'David', '2023-07-30', '88.5', 'False']
['5', 'Eva', '2023-10-01', '95.0', 'True']


In [10]:
with open("./data/sample_data.csv") as f:
    for line in f:
        print(line.strip("\n").split(","))

['id', 'name', 'signup_date', 'score', 'active']
['1', 'Alice', '2023-01-15', '85.5', 'True']
['2', 'Bob', '2023-03-22', '92.0', 'False']
['3', 'Charlie', '2023-05-10', '78.0', 'True']
['4', 'David', '2023-07-30', '88.5', 'False']
['5', 'Eva', '2023-10-01', '95.0', 'True']


In [39]:
def open_csv(filename, batch_size, delimiter, headers=None, hook=None):

    with open(filename, newline="") as f:
        if headers is None:
            headers = next(f).strip("\r\n")

        batch = list(islice(f, batch_size))

        while batch:
            if hook:
                batch = hook(batch)

            yield DictReader(batch, delimiter=delimiter, fieldnames=headers.split(delimiter))
            batch = list(islice(f, batch_size))


In [40]:
for batch in open_csv("./data/sample_data.csv", batch_size=2, delimiter=",", hook=None):
    for line in batch:
        print(line)

{'id': '1', 'name': 'Alice', 'signup_date': '2023-01-15', 'score': '85.5', 'active': 'True'}
{'id': '2', 'name': 'Bob', 'signup_date': '2023-03-22', 'score': '92.0', 'active': 'False'}
{'id': '3', 'name': 'Charlie', 'signup_date': '2023-05-10', 'score': '78.0', 'active': 'True'}
{'id': '4', 'name': 'David', 'signup_date': '2023-07-30', 'score': '88.5', 'active': 'False'}
{'id': '5', 'name': 'Eva', 'signup_date': '2023-10-01', 'score': '95.0', 'active': 'True'}


In [87]:
class CSVBatchReader:
    def __init__(self, filename, batch_size, delimiter=',', headers=None):
        self.filename = filename
        self.batch_size = batch_size
        self.delimiter = delimiter
        self.raw_headers = headers
        self.headers = headers
        self.file = None

    def __iter__(self):
        self.file = open(self.filename, newline='')
        if self.raw_headers is None:
            first_line = next(self.file).strip("\r\n")
            self.headers = first_line.split(self.delimiter)
        return self

    def __next__(self):
        batch_lines = list(islice(self.file, self.batch_size))

        # if batch_lines is an empty list close the file
        if not batch_lines:
            self.file.close()
            raise StopIteration

        return DictReader(batch_lines, delimiter=self.delimiter, fieldnames=self.headers)

Notice that this class does not behave as a typical iterator. I can iterate as many times I want without recreating the instance of the class

In [88]:
batches = CSVBatchReader(filename="./data/sample_data.csv", batch_size=2, delimiter=",")

In [89]:
for batch in batches:
    for line in batch:
        print(line)

{'id': '1', 'name': 'Alice', 'signup_date': '2023-01-15', 'score': '85.5', 'active': 'True'}
{'id': '2', 'name': 'Bob', 'signup_date': '2023-03-22', 'score': '92.0', 'active': 'False'}
{'id': '3', 'name': 'Charlie', 'signup_date': '2023-05-10', 'score': '78.0', 'active': 'True'}
{'id': '4', 'name': 'David', 'signup_date': '2023-07-30', 'score': '88.5', 'active': 'False'}
{'id': '5', 'name': 'Eva', 'signup_date': '2023-10-01', 'score': '95.0', 'active': 'True'}


In [90]:
all_lines = []
for batch in batches:
    lines = list(batch)
    all_lines.extend(lines)

In [91]:
all_lines

[{'id': '1',
  'name': 'Alice',
  'signup_date': '2023-01-15',
  'score': '85.5',
  'active': 'True'},
 {'id': '2',
  'name': 'Bob',
  'signup_date': '2023-03-22',
  'score': '92.0',
  'active': 'False'},
 {'id': '3',
  'name': 'Charlie',
  'signup_date': '2023-05-10',
  'score': '78.0',
  'active': 'True'},
 {'id': '4',
  'name': 'David',
  'signup_date': '2023-07-30',
  'score': '88.5',
  'active': 'False'},
 {'id': '5',
  'name': 'Eva',
  'signup_date': '2023-10-01',
  'score': '95.0',
  'active': 'True'}]

Assuming now that we want to apply a function before we pass the file into the DictReader. For example DictReader does not support double character delimiter but the received file uses #! as delimiter.

In [92]:
# Read the file into a string
with open("./data/sample_data.csv", "r", encoding="utf-8") as f:
    content = f.read()

# Replace delimiters
new_content = content.replace(",", "#!")

# Write the modified content to a new file
with open("./data/new_sample_data.csv", "w", encoding="utf-8", newline="") as f:
    f.write(new_content)


In [130]:

class CSVBatchReader:
    def __init__(self, filename, batch_size, delimiter=',', headers=None, hook=None):
        self.filename = filename
        self.batch_size = batch_size
        self.delimiter = delimiter
        self.raw_headers = headers
        self.headers = headers
        self.file = None
        self.hook = hook

    def __iter__(self):
        self.file = open(self.filename, newline='')
        if self.raw_headers is None:
            first_line = next(self.file).strip("\r\n")
            self.headers = first_line.split(self.delimiter)
        return self

    def __next__(self):

        batch_lines = list(islice(self.file, self.batch_size))

        # if batch_lines is an empty list close the file
        if not batch_lines:
            self.file.close()
            raise StopIteration


        return DictReader(batch_lines, delimiter=self.delimiter, fieldnames=self.headers)


    @validate_delimiter_change
    def replace_delimiter(self, lines, /, old_delimiter=None, new_delimiter=","):
        print('replace_delimiter was called ...')
        if old_delimiter is None:
            old_delimiter = self.delimiter
        return (line.replace(old_delimiter, new_delimiter) for line in lines)


In [131]:
batches = CSVBatchReader(filename="./data/new_sample_data.csv",
                         batch_size=2,
                         delimiter="#!",
                         hook=partial(validate_delimiter_change, old_delimiter="#!", new_delimiter=","))
for batch in batches:
    for line in batch:
        print(line)

functools.partial(<AdapterWrapper at 0x1048d3ed0 for function at 0x104b256c0>, old_delimiter='#!', new_delimiter=',') True
['1#!Alice#!2023-01-15#!85.5#!True\n', '2#!Bob#!2023-03-22#!92.0#!False\n']


TypeError: "delimiter" must be a 1-character string

In [111]:
from functools import reduce

l = ['this is the first line', 'this is the second line', 'this is the third line']

def custom_join(delimiter, items):
    if not items:
        return ''
    return reduce(lambda t_1, t_2: t_1 + delimiter + t_2 + "\n", items)

text = custom_join(",", l)

In [112]:
text

'this is the first line,this is the second line\n,this is the third line\n'

In [113]:
custom_join(", ", [])

''